🧠 Core Idea

To estimate grape volume:

1. Detect grapes (YOLO) → bounding boxes

2. Segment grapes (SAM2) → pixel masks

3. Estimate depth (Depth Anything v3) → depth per pixel

4. Combine:

    - Mask → which pixels belong to grapes

    - Depth → how far each pixel is

    - Camera model → convert pixels → real-world size

5. Approximate volume (e.g., voxel integration or sphere fitting)

In [3]:
# ==========================================
# GRAPE VOLUME ESTIMATION PIPELINE
# YOLO + SAM2 + Depth Anything v3
# Production-style, modular, optimized
# ==========================================

# --------- INSTALL (run once) ---------
# !pip install ultralytics opencv-python numpy scipy
# !pip install git+https://github.com/DepthAnything/Depth-Anything-V3

# --------- IMPORTS ---------
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from scipy.spatial import ConvexHull
import tkinter as tk
from tkinter import filedialog
import os # Imported to set the initial directory

# Depth Anything v3
from depth_anything_v3.dpt import DepthAnythingV3

# --------- DEVICE SETUP ---------
device = "cuda" if torch.cuda.is_available() else "cpu"

# --------- LOAD MODELS ---------
# YOLO model (replace with your trained weights if needed)
det_model = YOLO("yolov8n.pt")

# Depth model
depth_model = DepthAnythingV3.from_pretrained(
    "LiheYoung/depth_anything_v3_large"
).to(device).eval()

# --------- CONFIG ---------
class Config:
    # Camera intrinsics (ADJUST THESE)
    fx = 1000.0
    fy = 1000.0
    cx = None
    cy = None

    # Depth scaling (VERY IMPORTANT)
    depth_scale = 0.5  # meters (tune this!)

    # Volume method: "voxel" or "sphere"
    method = "voxel"

config = Config()

# --------- HELPERS ---------
def preprocess_frame(frame):
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def get_depth_map(frame_rgb):
    depth = depth_model.infer_image(frame_rgb)

    # normalize
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    # scale to metric
    depth = depth * config.depth_scale

    return depth


def get_masks(frame, results):
    """
    Placeholder for SAM2 integration
    Replace with your SAM2 mask generator
    """
    masks = []

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)

            mask = np.zeros(frame.shape[:2], dtype=np.uint8)
            mask[y1:y2, x1:x2] = 1  # TEMP (replace with SAM2)

            masks.append(mask)

    return masks


# --------- VOLUME METHODS ---------
def volume_voxel(mask, depth, fx, fy):
    Z = depth[mask > 0]

    if len(Z) < 10:
        return 0.0

    # histogram-based integration (stable)
    bins = np.linspace(Z.min(), Z.max(), 50)
    hist, edges = np.histogram(Z, bins)

    volume = 0.0

    for i in range(len(hist)):
        Z_mid = (edges[i] + edges[i + 1]) / 2
        count = hist[i]

        pixel_area = (Z_mid ** 2) / (fx * fy)
        dz = edges[i + 1] - edges[i]

        volume += count * pixel_area * dz

    return volume


def volume_sphere(mask, depth):
    area_pixels = np.sum(mask)

    if area_pixels == 0:
        return 0.0

    r_pix = np.sqrt(area_pixels / np.pi)

    # approximate scale using mean depth
    Z = np.mean(depth[mask > 0])
    scale = Z / config.fx

    r_real = r_pix * scale

    return (4 / 3) * np.pi * (r_real ** 3)


# --------- MAIN PIPELINE ---------
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if config.cx is None:
        config.cx = width / 2
    if config.cy is None:
        config.cy = height / 2

    total_volume = 0.0
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = preprocess_frame(frame)

        # --- Detection ---
        results = det_model(frame)

        # --- Segmentation ---
        masks = get_masks(frame, results)

        # --- Depth ---
        depth = get_depth_map(frame_rgb)

        frame_volume = 0.0

        for mask in masks:
            if config.method == "voxel":
                vol = volume_voxel(mask, depth, config.fx, config.fy)
            else:
                vol = volume_sphere(mask, depth)

            frame_volume += vol

        total_volume += frame_volume
        frame_count += 1

        print(f"Frame {frame_count}: Volume = {frame_volume:.6f} m^3")

    cap.release()

    avg_volume = total_volume / max(frame_count, 1)

    print("\n===== RESULTS =====")
    print(f"Average Volume per Frame: {avg_volume:.6f} m^3")

    return avg_volume

OSError: [WinError 127] The specified procedure could not be found. Error loading "C:\Users\luebs\anaconda3\envs\dev\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [ ]:
def select_file():
    """Opens a file dialog and returns the selected file path."""
    # Define file types to filter the selection
    filetypes = (
        ('MP4 files', '*.MP4'),
        ('All files', '*.*')
    )

    filename = filedialog.askopenfilename(
        title='Open a file',
        initialdir=os.getcwd(), # Set the initial directory to current working directory
        filetypes=filetypes
    )
    
    if filename:
        print(f"Selected: {filename}")

    return filename

# 1. Create the main application window
root = tk.Tk()
root.title('File Chooser Example')
root.geometry('300x150') # Set the window size

# Optional: Hide the main window if only the dialog is needed immediately
# root.withdraw() 

# 2. Create the "Choose File" button
open_button = tk.Button(
    root,
    text='Choose File',
    command=select_file # Link the button to the select_file function
)

# 3. Place the button in the window
open_button.pack(expand=True, pady=20) # Use pack to display the button

# 4. Run the application's main event loop
root.mainloop()

# --------- RUN ---------
if __name__ == "__main__":
    video_path = filename  # CHANGE THIS
    process_video(video_path)